# SMETANA and SteadyCom analysis

In [1]:
import reframed
import pandas as pd

from os import listdir,makedirs
import subprocess
from pathlib import Path
import sys
sys.path.append("../functions/")
from simulation import general
from simulation import steadycom_functions

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
reframed.set_default_solver("cplex")

## SMETANA

### Run SMETANA

## SteadyCom

#### Load data and models

**Load models**

*E. coli mutants  (top 7)*

In [4]:
models_ecoli = general.load_models_guillen2021(all_38_strains=False)

INFO     | Loading models from Guillen et al (2021)...


*Gut microbiome members*

In [5]:
models_gut = general.load_models_venturelli2018()

INFO     | Loading models from Venturelli et al (2018)...


**Load media**

*For E. coli mutants (M9 medium to force growth of specific members)*

In [6]:
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")
compounds = list(media_db[media_db.medium=="M9"].compound.values)
env_ecoli = reframed.Environment.from_compounds(compounds,max_uptake=10)

*For gut microbiome members*

In [7]:
media_db_gut = pd.read_csv("../data/Venturelli2018/generated/media_db.tsv",sep="\t")
reactions = media_db_gut[media_db_gut.medium=="LB_extend"].compound.values
#reactions = [rxn.replace("40","91").replace("41","93") for rxn in reactions]

env_gut = reframed.Environment.from_reactions(reactions,max_uptake=10)

**Load other data**

In [8]:
gene_id2name = pd.read_csv("../data/Guillen2021/generated/gene_id2name.tsv",sep="\t")

**Define communities and constraints for abundance**

In [9]:

# E. coli community
relative_abundance_df_ecoli = pd.read_csv("../data/Guillen2021/generated/communities_steadiercom.tsv",
                                    sep="\t",
                                    header=None,
                                    names=["community_id", "member", "abundance"])

# Gut microbes community (multiple files)
relative_abundance_dfs = []
for file in listdir("../data/Venturelli2018/generated/communities_steadiercom/"):
    if not file.endswith(".tsv"):
        continue
    relative_abundance_df = pd.read_csv("../data/Venturelli2018/generated/communities_steadiercom/"+file,
                                             sep="\t",
                                             header=None,
                                             names=["community_id", "member", "abundance"])
    relative_abundance_dfs.append(relative_abundance_df)

# Gut microbes - pairwise
relative_abundance_df_gut = pd.concat(relative_abundance_dfs)
relative_abundance_df_gut_pairwise = pd.read_csv( "../data/Venturelli2018/generated/communities_pairwise.tsv",
    sep="\t", header=None, names=["community_id", "member", "abundance"]
)

# Make constraints
constraints_ecoli = steadycom_functions.build_constraints_from_abundance(relative_abundance_df_ecoli,prefix="x_")
constraints_gut = steadycom_functions.build_constraints_from_abundance(relative_abundance_df_gut)
constraints_gut_pairwise = steadycom_functions.build_constraints_from_abundance(relative_abundance_df_gut_pairwise)


**Make directories for output**

In [10]:
gut_dir_fixed = "../output/Venturelli2018/steadycom/fixed_abundance/"
gut_dir_not_fixed = "../output/Venturelli2018/steadycom/no_abundance/"
gut_dir_pairwise_fixed = "../output/Venturelli2018/steadycom/pairwise_fixed_abundance/"
gut_dir_pairwise_not_fixed = "../output/Venturelli2018/steadycom/pairwise_no_abundance/"
ecoli_dir_fixed =  "../output/Guillen2021/steadycom/fixed_abundance/"
ecoli_dir_not_fixed =  "../output/Guillen2021/steadycom/no_abundance/"

list_of_dirs = [gut_dir_fixed,gut_dir_not_fixed,gut_dir_pairwise_fixed,gut_dir_pairwise_not_fixed,ecoli_dir_fixed,ecoli_dir_not_fixed]
for dir_ in list_of_dirs:
    makedirs(dir_,exist_ok=True)

### Run SteadyCom

#### Run SteadyCom for E.coli with fixed abundance

In [11]:
dir_ = ecoli_dir_fixed
cross_feeding_comms_df,relative_abundance_comms_df,growth_comms_df = steadycom_functions.SteadyComCommunities(models_ecoli,
                                                                                 constraints_ecoli,
                                                                                 env_ecoli,
                                                                                 fixed_relative_abundance=True)

# Save results
cross_feeding_comms_df.to_csv(Path(dir_).joinpath("cross_feeding.tsv"),sep="\t",index=None)
relative_abundance_comms_df.to_csv(Path(dir_).joinpath("relative_abundance.tsv"),sep="\t")
growth_comms_df.to_csv(Path(dir_).joinpath("growth.tsv"),sep="\t")

Started SteadyCom analysis for Ecoli_mutants
	Running SteadyCom took: 28.99 minutes


#### Run SteadyCom for E.coli 

In [12]:
dir_ = ecoli_dir_not_fixed
cross_feeding_comms_df,relative_abundance_comms_df,growth_comms_df = steadycom_functions.SteadyComCommunities(models_ecoli,
                                                                                 constraints_ecoli,
                                                                                 env_ecoli,
                                                                                 fixed_relative_abundance=False)

# Save results
cross_feeding_comms_df.to_csv(Path(dir_).joinpath("cross_feeding.tsv"),sep="\t",index=None)
relative_abundance_comms_df.to_csv(Path(dir_).joinpath("relative_abundance.tsv"),sep="\t")
growth_comms_df.to_csv(Path(dir_).joinpath("growth.tsv"),sep="\t")

Started SteadyCom analysis for Ecoli_mutants
	Running SteadyCom took: 166.50 minutes


#### Run SteadyCom for Gut microbiome

#### Fixed abundance

In [13]:
dir_ = gut_dir_fixed
cross_feeding_comms_df,relative_abundance_comms_df,growth_comms_df = steadycom_functions.SteadyComCommunities(models_gut,
                                                                                          constraints_gut,
                                                                                          env_gut,
                                                                                          fixed_relative_abundance=True)

# Save results
cross_feeding_comms_df.to_csv(Path(dir_).joinpath("cross_feeding.tsv"),sep="\t",index=None)
relative_abundance_comms_df.to_csv(Path(dir_).joinpath("relative_abundance.tsv"),sep="\t")
growth_comms_df.to_csv(Path(dir_).joinpath("growth.tsv"),sep="\t")

Started SteadyCom analysis for ER-
	Running SteadyCom took: 0.20 minutes


/Users/idunmariaburgos/SteadierCom/reframed/reframed/core/elements.py:62: UserWarning: Atomic weight not listed for elements: {'X'}
  warn(f"Atomic weight not listed for elements: {missing}")


Started SteadyCom analysis for DP-
	Running SteadyCom took: 0.21 minutes
Started SteadyCom analysis for PC-
	Running SteadyCom took: 0.21 minutes
Started SteadyCom analysis for EL-
	Running SteadyCom took: 0.21 minutes
Started SteadyCom analysis for BT-
	Running SteadyCom took: 0.19 minutes
Started SteadyCom analysis for BU-
	Running SteadyCom took: 0.20 minutes
Started SteadyCom analysis for NONE
	Running SteadyCom took: 0.21 minutes
Started SteadyCom analysis for BV-
	Running SteadyCom took: 0.19 minutes
Started SteadyCom analysis for CH-
	Running SteadyCom took: 0.21 minutes
Started SteadyCom analysis for BO-
	Running SteadyCom took: 0.20 minutes
Started SteadyCom analysis for BH-
	Running SteadyCom took: 0.21 minutes
Started SteadyCom analysis for FP-
	Running SteadyCom took: 0.19 minutes
Started SteadyCom analysis for CA-
	Running SteadyCom took: 0.21 minutes


#### Not fixed abundance

In [14]:
dir_ = gut_dir_not_fixed
cross_feeding_comms_df,relative_abundance_comms_df,growth_comms_df = steadycom_functions.SteadyComCommunities(models_gut,
                                                                                          constraints_gut,
                                                                                          env_gut,
                                                                                          fixed_relative_abundance=False)

# Save results
cross_feeding_comms_df.to_csv(Path(dir_).joinpath("cross_feeding.tsv"),sep="\t",index=None)
relative_abundance_comms_df.to_csv(Path(dir_).joinpath("relative_abundance.tsv"),sep="\t")
growth_comms_df.to_csv(Path(dir_).joinpath("growth.tsv"),sep="\t")

Started SteadyCom analysis for ER-
	Running SteadyCom took: 0.29 minutes


/Users/idunmariaburgos/SteadierCom/reframed/reframed/core/elements.py:62: UserWarning: Atomic weight not listed for elements: {'X'}
  warn(f"Atomic weight not listed for elements: {missing}")


Started SteadyCom analysis for DP-
	Running SteadyCom took: 0.30 minutes
Started SteadyCom analysis for PC-
	Running SteadyCom took: 0.30 minutes
Started SteadyCom analysis for EL-
	Running SteadyCom took: 0.30 minutes
Started SteadyCom analysis for BT-
	Running SteadyCom took: 0.28 minutes
Started SteadyCom analysis for BU-
	Running SteadyCom took: 0.29 minutes
Started SteadyCom analysis for NONE
	Running SteadyCom took: 0.32 minutes
Started SteadyCom analysis for BV-
	Running SteadyCom took: 0.28 minutes
Started SteadyCom analysis for CH-
	Running SteadyCom took: 0.32 minutes
Started SteadyCom analysis for BO-
	Running SteadyCom took: 0.29 minutes
Started SteadyCom analysis for BH-
	Running SteadyCom took: 0.32 minutes
Started SteadyCom analysis for FP-
	Running SteadyCom took: 0.31 minutes
Started SteadyCom analysis for CA-
	Running SteadyCom took: 0.34 minutes


##### Fixed abundance - pairwise

In [15]:
dir_ = gut_dir_pairwise_fixed
cross_feeding_comms_df,relative_abundance_comms_df,growth_comms_df = steadycom_functions.SteadyComCommunities(models_gut,
                                                                                          constraints_gut_pairwise,
                                                                                          env_gut,
                                                                                          fixed_relative_abundance=True)

# Save results
cross_feeding_comms_df.to_csv(Path(dir_).joinpath("cross_feeding.tsv"),sep="\t",index=None)
relative_abundance_comms_df.to_csv(Path(dir_).joinpath("relative_abundance.tsv"),sep="\t")
growth_comms_df.to_csv(Path(dir_).joinpath("growth.tsv"),sep="\t")

Started SteadyCom analysis for BH-CA
	Running SteadyCom took: 0.01 minutes
Started SteadyCom analysis for BU-BH
	Running SteadyCom took: 0.01 minutes
Started SteadyCom analysis for PC-BH


/Users/idunmariaburgos/SteadierCom/reframed/reframed/core/elements.py:62: UserWarning: Atomic weight not listed for elements: {'X'}
  warn(f"Atomic weight not listed for elements: {missing}")


	Running SteadyCom took: 0.01 minutes
Started SteadyCom analysis for BO-BH
	Running SteadyCom took: 0.01 minutes
Started SteadyCom analysis for BV-BH
	Running SteadyCom took: 0.01 minutes
Started SteadyCom analysis for BT-BH
	Running SteadyCom took: 0.01 minutes
Started SteadyCom analysis for EL-BH
	Running SteadyCom took: 0.02 minutes
Started SteadyCom analysis for FP-BH
	Running SteadyCom took: 0.02 minutes
Started SteadyCom analysis for CH-BH
	Running SteadyCom took: 0.01 minutes
Started SteadyCom analysis for DP-BH
	Running SteadyCom took: 0.01 minutes
Started SteadyCom analysis for ER-BH
	Running SteadyCom took: 0.02 minutes
Started SteadyCom analysis for BU-CA
	Running SteadyCom took: 0.02 minutes
Started SteadyCom analysis for PC-CA
	Running SteadyCom took: 0.02 minutes
Started SteadyCom analysis for BO-CA
	Running SteadyCom took: 0.03 minutes
Started SteadyCom analysis for BV-CA
	Running SteadyCom took: 0.01 minutes
Started SteadyCom analysis for BT-CA
	Running SteadyCom took: 

##### Not fixed abundance - pairwise

In [16]:
dir_ = gut_dir_pairwise_not_fixed
cross_feeding_comms_df,relative_abundance_comms_df,growth_comms_df = steadycom_functions.SteadyComCommunities(models_gut,
                                                                                          constraints_gut_pairwise,
                                                                                          env_gut,
                                                                                          fixed_relative_abundance=False)

# Save results
cross_feeding_comms_df.to_csv(Path(dir_).joinpath("cross_feeding.tsv"),sep="\t",index=None)
relative_abundance_comms_df.to_csv(Path(dir_).joinpath("relative_abundance.tsv"),sep="\t")
growth_comms_df.to_csv(Path(dir_).joinpath("growth.tsv"),sep="\t")

Started SteadyCom analysis for BH-CA
	Running SteadyCom took: 0.02 minutes
Started SteadyCom analysis for BU-BH
	Running SteadyCom took: 0.03 minutes
Started SteadyCom analysis for PC-BH


/Users/idunmariaburgos/SteadierCom/reframed/reframed/core/elements.py:62: UserWarning: Atomic weight not listed for elements: {'X'}
  warn(f"Atomic weight not listed for elements: {missing}")


	Running SteadyCom took: 0.03 minutes
Started SteadyCom analysis for BO-BH
	Running SteadyCom took: 0.03 minutes
Started SteadyCom analysis for BV-BH
	Running SteadyCom took: 0.04 minutes
Started SteadyCom analysis for BT-BH
	Running SteadyCom took: 0.04 minutes
Started SteadyCom analysis for EL-BH
	Running SteadyCom took: 0.02 minutes
Started SteadyCom analysis for FP-BH
	Running SteadyCom took: 0.03 minutes
Started SteadyCom analysis for CH-BH
	Running SteadyCom took: 0.02 minutes
Started SteadyCom analysis for DP-BH
	Running SteadyCom took: 0.03 minutes
Started SteadyCom analysis for ER-BH
	Running SteadyCom took: 0.03 minutes
Started SteadyCom analysis for BU-CA
	Running SteadyCom took: 0.03 minutes
Started SteadyCom analysis for PC-CA
	Running SteadyCom took: 0.02 minutes
Started SteadyCom analysis for BO-CA
	Running SteadyCom took: 0.03 minutes
Started SteadyCom analysis for BV-CA
	Running SteadyCom took: 0.03 minutes
Started SteadyCom analysis for BT-CA
	Running SteadyCom took: 